# GLiClass on the full Bloomberg corpus (fin-decider)

Runtime → Change runtime type → **T4 GPU**. Upload `fin-decider-colab.tar.gz` to `MyDrive/fin-decider/` first.

Every run uses `--resume`, so after a disconnect or the ~12h session limit just run all cells again: finished articles are skipped. Results live on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/fin-decider'
RESULTS = f'{DRIVE}/classification_results'

In [ ]:
%%bash
set -e
rm -rf /content/pkg /content/fin-decider && mkdir -p /content/pkg
tar -xzf /content/drive/MyDrive/fin-decider/fin-decider-colab.tar.gz -C /content/pkg
cat /content/pkg/bundle.json
BRANCH=$(python -c "import json;print(json.load(open('/content/pkg/bundle.json'))['branch'])")
git clone -q -b "$BRANCH" /content/pkg/repo.bundle /content/fin-decider
mkdir -p /content/fin-decider/corpus && cp /content/pkg/canonical.parquet /content/pkg/manifest.json /content/fin-decider/corpus/
cd /content/fin-decider && git log -1 --oneline

In [ ]:
%%bash
cd /content/fin-decider
pip install -q -r /content/pkg/requirements-colab.txt
pip install -q --no-deps -e .
python -c "import torch;print(torch.__version__, torch.cuda.get_device_name(0))"

## 1. Throughput gate (200 articles)
Prints articles/s; the full corpus is 445,391 articles. Check the projection before the full run.

In [ ]:
%%bash -s "$RESULTS"
cd /content/fin-decider
classify run -m gliclass --limit 200 --resume --output-dir "$1" --progress-every 50 --set run.allow_dirty_git=false

## 2. Full corpus
Safe to re-run after any disconnect.

In [ ]:
%%bash -s "$RESULTS"
cd /content/fin-decider
classify run -m gliclass --resume --output-dir "$1" --progress-every 2000